In [1]:
# DAY 3: DATA CLEANING PIPELINE
# FreshBite Bakery - Inventory Waste Analysis

import pandas as pd
import numpy as np

print("="*60)
print("DATA CLEANING - FRESHBITE BAKERY")
print("="*60)


DATA CLEANING - FRESHBITE BAKERY


In [2]:
# LOAD RAW DATA

print("\n[1/6] Loading raw data...")

products = pd.read_csv('products.csv')
sales = pd.read_csv('sales_transactions.csv')
inventory = pd.read_csv('inventory_records.csv')

print(f"✓ Products: {len(products)} rows")
print(f"✓ Sales: {len(sales)} rows")
print(f"✓ Inventory: {len(inventory)} rows")


[1/6] Loading raw data...
✓ Products: 45 rows
✓ Sales: 21576 rows
✓ Inventory: 16437 rows


In [3]:
# CLEAN PRODUCTS TABLE

print("\n" + "="*60)
print("[2/6] CLEANING PRODUCTS TABLE")
print("="*60)

# Issue: Missing shelf_life_days
print("\n--- Fixing Missing shelf_life_days ---")
print(f"Missing before: {products['shelf_life_days'].isnull().sum()}")



[2/6] CLEANING PRODUCTS TABLE

--- Fixing Missing shelf_life_days ---
Missing before: 4


In [4]:
# Fill missing values based on category averages
category_avg_shelf = products.groupby('category')['shelf_life_days'].mean()
print("\nCategory average shelf life:")
print(category_avg_shelf)


Category average shelf life:
category
Bagel       3.000000
Bar         3.500000
Bread       2.866667
Cake        2.000000
Cookie      5.000000
Cupcake     2.000000
Muffin      3.000000
Pastry      1.428571
Pie         3.000000
Sandwich    1.000000
Savory      1.000000
Name: shelf_life_days, dtype: float64


In [5]:
# Fill missing with category average
for idx, row in products.iterrows():
    if pd.isnull(row['shelf_life_days']):
        category = row['category']
        products.at[idx, 'shelf_life_days'] = round(category_avg_shelf[category])

print(f"Missing after: {products['shelf_life_days'].isnull().sum()}")


Missing after: 0


In [6]:
# Issue: Standardize product names
print("\n--- Standardizing Product Names ---")
print("Before:")
print(products[products['product_name'].str.contains('Sourdough', case=False)][['product_id', 'product_name']])



--- Standardizing Product Names ---
Before:
  product_id     product_name
0       P001  Sourdough Bread
1       P002        Sourdough


In [8]:
# Replace 'Sourdough' with 'Sourdough Bread'
products['product_name'] = products['product_name'].replace('Sourdough', 'Sourdough Bread')

print("\nAfter:")
print(products[products['product_name'].str.contains('Sourdough', case=False)][['product_id', 'product_name']])





After:
  product_id     product_name
0       P001  Sourdough Bread
1       P002  Sourdough Bread


In [9]:
# Add profit margin column
products['profit_per_unit'] = products['selling_price'] - products['unit_cost']

print("\n✓ Products table cleaned")
print(f"Final shape: {products.shape}")



✓ Products table cleaned
Final shape: (45, 7)


In [10]:
# CLEAN SALES TABLE

print("\n" + "="*60)
print("[3/6] CLEANING SALES TABLE")
print("="*60)

# Convert date to datetime
sales['sale_date'] = pd.to_datetime(sales['sale_date'])



[3/6] CLEANING SALES TABLE


In [11]:
# Issue: Missing store_location
print("\n--- Fixing Missing store_location ---")
print(f"Missing before: {sales['store_location'].isnull().sum()}")



--- Fixing Missing store_location ---
Missing before: 425


In [12]:
# Strategy: Remove rows with missing store (small percentage)
sales_clean = sales[sales['store_location'].notna()].copy()

print(f"Missing after: {sales_clean['store_location'].isnull().sum()}")
print(f"Rows removed: {len(sales) - len(sales_clean)}")

Missing after: 0
Rows removed: 425


In [13]:
# Add time-based features
sales_clean['month'] = sales_clean['sale_date'].dt.month
sales_clean['day_of_week'] = sales_clean['sale_date'].dt.dayofweek
sales_clean['is_weekend'] = sales_clean['day_of_week'].isin([5, 6]).astype(int)

print("\n✓ Sales table cleaned")
print(f"Final shape: {sales_clean.shape}")


✓ Sales table cleaned
Final shape: (21151, 8)


In [14]:
# CLEAN INVENTORY TABLE

print("\n" + "="*60)
print("[4/6] CLEANING INVENTORY TABLE")
print("="*60)

# Convert date to datetime
inventory['record_date'] = pd.to_datetime(inventory['record_date'])

# Issue 1: Negative spoilage values
print("\n--- Fixing Negative Spoilage ---")
negative_count = (inventory['stock_spoiled'] < 0).sum()
print(f"Negative values found: {negative_count}")


[4/6] CLEANING INVENTORY TABLE

--- Fixing Negative Spoilage ---
Negative values found: 178


In [15]:
# Replace negative with 0
inventory.loc[inventory['stock_spoiled'] < 0, 'stock_spoiled'] = 0
print(f"Negative values after fix: {(inventory['stock_spoiled'] < 0).sum()}")


Negative values after fix: 0


In [16]:
# Issue 2: Outlier spoilage values
print("\n--- Fixing Outlier Spoilage ---")

# Calculate IQR for outlier detection
Q1 = inventory['stock_spoiled'].quantile(0.25)
Q3 = inventory['stock_spoiled'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 3 * IQR

print(f"Upper bound for spoilage: {upper_bound:.0f}")
outlier_count = (inventory['stock_spoiled'] > upper_bound).sum()
print(f"Outliers found: {outlier_count}")

# Cap outliers at upper bound
inventory.loc[inventory['stock_spoiled'] > upper_bound, 'stock_spoiled'] = upper_bound
print(f"Outliers after fix: {(inventory['stock_spoiled'] > upper_bound).sum()}")



--- Fixing Outlier Spoilage ---
Upper bound for spoilage: 13
Outliers found: 156
Outliers after fix: 0


In [17]:
# Issue 3: Missing spoilage values
print("\n--- Fixing Missing Spoilage ---")
print(f"Missing before: {inventory['stock_spoiled'].isnull().sum()}")

# Fill missing with median spoilage
median_spoilage = inventory['stock_spoiled'].median()
inventory['stock_spoiled'].fillna(median_spoilage, inplace=True)

print(f"Missing after: {inventory['stock_spoiled'].isnull().sum()}")
print(f"Filled with median: {median_spoilage}")

# Calculate total stock and spoilage rate
inventory['total_stock'] = inventory['stock_start_of_day'] + inventory['stock_received']
inventory['spoilage_rate'] = (inventory['stock_spoiled'] / inventory['total_stock'] * 100).round(2)

# Handle division by zero (when total_stock = 0)
inventory['spoilage_rate'].fillna(0, inplace=True)

print("\n✓ Inventory table cleaned")
print(f"Final shape: {inventory.shape}")


--- Fixing Missing Spoilage ---
Missing before: 821
Missing after: 0
Filled with median: 2.0

✓ Inventory table cleaned
Final shape: (16437, 9)


C:\Users\deeks\AppData\Local\Temp\ipykernel_7532\808000181.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  inventory['stock_spoiled'].fillna(median_spoilage, inplace=True)
C:\Users\deeks\AppData\Local\Temp\ipykernel_7532\808000181.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a c

In [18]:
# CREATE MERGED DATASET
# ============================================
print("\n" + "="*60)
print("[5/6] CREATING MERGED ANALYSIS DATASET")
print("="*60)

# Merge inventory with products
inventory_products = inventory.merge(products, on='product_id', how='left')

# Calculate waste cost
inventory_products['waste_cost'] = inventory_products['stock_spoiled'] * inventory_products['unit_cost']

print(f"✓ Merged inventory + products: {inventory_products.shape}")

# Aggregate sales by product and store
sales_summary = sales_clean.groupby(['product_id', 'store_location']).agg({
    'quantity_sold': 'sum',
    'transaction_id': 'count'
}).reset_index()

sales_summary.columns = ['product_id', 'store_location', 'total_quantity_sold', 'transaction_count']

print(f"✓ Sales summary created: {sales_summary.shape}")


[5/6] CREATING MERGED ANALYSIS DATASET
✓ Merged inventory + products: (16437, 16)
✓ Sales summary created: (135, 4)


In [20]:
# SAVE CLEANED DATA
# ============================================
print("\n" + "="*60)
print("[6/6] SAVING CLEANED DATA")
print("="*60)

# Save individual cleaned tables
products.to_csv('products_clean.csv', index=False)
print("✓ Saved: products_clean.csv")

sales_clean.to_csv('sales_clean.csv', index=False)
print("✓ Saved: sales_clean.csv")

inventory.to_csv('inventory_clean.csv', index=False)
print("✓ Saved: inventory_clean.csv")

# Save merged datasets
inventory_products.to_csv('inventory_with_products.csv', index=False)
print("✓ Saved: inventory_with_products.csv")

sales_summary.to_csv('sales_summary.csv', index=False)
print("✓ Saved: sales_summary.csv")


[6/6] SAVING CLEANED DATA
✓ Saved: products_clean.csv
✓ Saved: sales_clean.csv
✓ Saved: inventory_clean.csv
✓ Saved: inventory_with_products.csv
✓ Saved: sales_summary.csv


In [22]:
# CLEANING SUMMARY
# ============================================
print("\n" + "="*60)
print("CLEANING COMPLETE - SUMMARY")
print("="*60)

print("\n📊 CLEANED DATASETS:")
print(f"   • products_clean.csv: {len(products)} rows")
print(f"   • sales_clean.csv: {len(sales_clean)} rows")
print(f"   • inventory_clean.csv: {len(inventory)} rows")
print(f"   • inventory_with_products.csv: {len(inventory_products)} rows")
print(f"   • sales_summary.csv: {len(sales_summary)} rows")

print("\n✅ FIXES APPLIED:")
print("   • Filled missing shelf_life using category averages")
print("   • Standardized 'Sourdough' product name")
print("   • Removed sales with missing store_location")
print("   • Fixed negative spoilage values → 0")
print("   • Capped outlier spoilage at upper bound")
print("   • Filled missing spoilage with median")
print("   • Added calculated columns (profit, spoilage_rate, waste_cost)")




CLEANING COMPLETE - SUMMARY

📊 CLEANED DATASETS:
   • products_clean.csv: 45 rows
   • sales_clean.csv: 21151 rows
   • inventory_clean.csv: 16437 rows
   • inventory_with_products.csv: 16437 rows
   • sales_summary.csv: 135 rows

✅ FIXES APPLIED:
   • Filled missing shelf_life using category averages
   • Standardized 'Sourdough' product name
   • Removed sales with missing store_location
   • Fixed negative spoilage values → 0
   • Capped outlier spoilage at upper bound
   • Filled missing spoilage with median
   • Added calculated columns (profit, spoilage_rate, waste_cost)
